Retail HOL - Talk to Your Data with Semantic Views and Cortex Agent
*Co-authored with CoCo*

# Talk to Your Retail Data — Semantic View + Cortex Agent (Natural Language Analytics)

---

## Previously...

In **Notebook 03**, you built an Dynamic Iceberg Tables( Silver/Gold), AI-enriched open-lakehouse pipeline:

```
PostgreSQL → OpenFlow CDC → Iceberg Bronze → Dynamic Iceberg Tables (Silver/Gold)
                                                 ↑ ADAPTIVE refresh
                                                 ↑ Frozen Regions
                                                 ↑ Cortex AI inline enrichment
```

The pipeline works. The Gold tables are refreshing. But your VP of Retail just said:

> *"I'm not going to learn SQL. I just want to ask: 'What happened this week?' and get an answer."*

This notebook makes that happen.

---

## What We're Building

```
Gold Tables → Semantic View (business logic) → Cortex Agent (NL interface) → Anyone asks questions
```

| Component | Purpose | Snowflake Feature |
|-----------|---------|-------------------|
| **Semantic View** | Defines business meaning of columns, metrics, and relationships | `CREATE SEMANTIC VIEW` |
| **Cortex Agent** | AI that understands your data model and generates SQL | `CREATE AGENT` |
| **Natural Language Queries** | Ask questions in English, get answers from data | `SNOWFLAKE.CORTEX.DATA_AGENT_RUN` |

---

## What You'll Learn

| # | Skill | Feature |
|---|-------|----------|
| 1 | Define a **Semantic View** that teaches AI the business meaning of your data | Semantic View YAML |
| 2 | Add **verified queries** as guardrails for common questions | VQRs |
| 3 | Create a **Cortex Agent** with custom instructions and persona | `CREATE AGENT` |
| 4 | Register the agent with **Snowflake Intelligence (CoWork)** for team access | `ALTER SNOWFLAKE INTELLIGENCE` |
| 5 | Ask **cross-domain questions** that span multiple Gold tables | `DATA_AGENT_RUN` |

**Time estimate:** ~10 minutes to run all cells

In [ ]:
%%sql -r dataframe_1
-- Setup
USE ROLE RETAIL_HOL_ROLE;
USE WAREHOUSE RETAIL_HOL_WH;
USE DATABASE RETAIL_HOL_ICEBERG;
USE SCHEMA GOLD_MANAGED_ICEBERG;

---

## Quick Look: What's in our Gold Layer?

Before teaching an AI about our data, let's remind ourselves what's there. These are the Gold tables from Notebook 03 — each tuned to a different decision persona's.

In [ ]:
%%sql -r dataframe_2
-- What Gold tables do we have?
SHOW DYNAMIC TABLES IN SCHEMA GOLD_MANAGED_ICEBERG;

In [ ]:
%%sql -r dataframe_3
-- Quick peek: daily product performance (the operational table)
SELECT PRODUCT_NAME, PERFORMANCE_DATE, TOTAL_UNITS_SOLD, STOCK_HEALTH_STATUS,
       ROUND(AVG_DAILY_SENTIMENT, 3) AS SENTIMENT
FROM GOLD_MANAGED_ICEBERG.DAILY_PRODUCT_PERFORMANCE
ORDER BY PERFORMANCE_DATE DESC
LIMIT 5;

The data is there — sales, inventory health, sentiment scores, all refreshing automatically. But to ask questions in natural language, an AI needs to understand:
- What each column *means* in business terms
- How tables *relate* to each other
- What *metrics* to compute (e.g., "online mix %" = online units / total units)
- What *verified answers* look like for common questions

That's exactly what a **Semantic View** provides.

---
## Step 1: Create the Semantic View

### The Business Context: 4 Tables, 4 Decision Altitudes

Imagine you run a 12-SKU candle and fragrance brand — products like *Frosted Juniper* and *Vanilla Cedarlight* — sold online and through 4 regional store networks, fulfilled out of 5 warehouses (LA, Houston, NYC, Chicago, SF). Every day, four different people ask you four different kinds of questions, at four different altitudes:

- **The merchandiser**, every morning: *"What sold yesterday, and is anything running low?"*
- **The VP of Retail**, every Monday: *"How did we do last week, and where are we trending?"*
- **The CX lead**, constantly: *"What are customers actually saying about us right now?"*
- **The supply chain planner**, constantly: *"Where physically is my inventory, and which warehouse needs a truck?"*

No single table answers all four questions well. So we model **4 tables**, each tuned to a different business altitude:

| Table | Grain | Decision It Powers |
|---|---|---|
| `DAILY_PRODUCT_PERFORMANCE` | 1 row / product / day | "Do I need to act today?" |
| `WEEKLY_PRODUCT_SCORECARD` | 1 row / product / week | "Are we improving or declining?" |
| `CUSTOMER_VOICE_ANALYTICS` | 1 row / comment | "Why did sentiment drop?" |
| `SILVER_INVENTORY` | 1 row / SKU / warehouse | "Which warehouse is critical?" |

The Semantic View teaches the agent how these four altitudes **connect** — so a question like *"Why did Frosted Juniper sales drop in the Northeast?"* pulls from the weekly trend, the daily detail, the warehouse data, and customer voice **all at once**.

### Anatomy of the Semantic View YAML

The YAML below defines:

| Section | Purpose | Example |
|---------|---------|--------|
| `tables` | Which physical tables to expose | `DAILY_PRODUCT_PERFORMANCE` |
| `dimensions` | Columns you GROUP BY or filter on | `PRODUCT_NAME`, `STOCK_HEALTH_STATUS` |
| `time_dimensions` | Date/time columns for time-series analysis | `PERFORMANCE_DATE`, `WEEK_START` |
| `facts` | Numeric columns (raw values) | `TOTAL_UNITS_SOLD`, `SENTIMENT_SCORE` |
| `metrics` | Pre-defined calculations (aggregations) | `ONLINE_MIX_PERCENTAGE`, `COMPLAINT_RATE` |
| `verified_queries` | Known-good SQL for common questions (guardrails) | "Top 5 products by sales" |

In [ ]:
%%sql -r dataframe_4
-- Create Semantic View using YAML specification
CALL SYSTEM$CREATE_SEMANTIC_VIEW_FROM_YAML(
  'RETAIL_HOL_ICEBERG.GOLD_MANAGED_ICEBERG',
  $$name: RETAIL_PRODUCT_INTELLIGENCE
description: >-
  Retail product intelligence model covering daily and weekly performance metrics,
  inventory health, customer sentiment, voice-of-customer analytics, and warehouse
  geolocation for a candle and fragrance retail business with 12 products sold across
  online and 4 regional stores. Warehouses in LA, Houston, NYC, Chicago, SF.

tables:
  - name: DAILY_PRODUCT_PERFORMANCE
    base_table:
      database: RETAIL_HOL_ICEBERG
      schema: GOLD_MANAGED_ICEBERG
      table: DAILY_PRODUCT_PERFORMANCE
    description: >-
      Daily product-level performance combining sales units, inventory position,
      stock health, quality scores, and customer review sentiment.
      Each row is one product on one day. Date range 2024.
    dimensions:
      - name: PRODUCT_NAME
        description: Product name such as Frosted Juniper, Citrus Petal, Lavender Honeycomb
        expr: PRODUCT_NAME
        data_type: VARCHAR
      - name: STOCK_HEALTH_STATUS
        description: Inventory health CRITICAL LOW NORMAL or OVERSTOCKED
        expr: STOCK_HEALTH_STATUS
        data_type: VARCHAR
    time_dimensions:
      - name: PERFORMANCE_DATE
        description: The calendar date of the performance record
        expr: PERFORMANCE_DATE
        data_type: DATE
    facts:
      - name: TOTAL_UNITS_SOLD
        description: Total units sold across all channels for that day
        expr: TOTAL_UNITS_SOLD
        data_type: NUMBER
      - name: ONLINE_UNITS
        description: Units sold via the online e-commerce channel
        expr: ONLINE_UNITS
        data_type: NUMBER
      - name: STORE_UNITS
        description: Units sold across all physical stores combined
        expr: STORE_UNITS
        data_type: NUMBER
      - name: INVENTORY_POSITION
        description: Total inventory stock available across all locations
        expr: INVENTORY_POSITION
        data_type: NUMBER
      - name: QUALITY_SCORE
        description: Product quality score from supplier audit 0 to 100
        expr: QUALITY_SCORE
        data_type: FLOAT
      - name: AVG_DAILY_SENTIMENT
        description: Average daily customer review sentiment score positive means good
        expr: AVG_DAILY_SENTIMENT
        data_type: FLOAT
      - name: POSITIVE_REVIEWS
        description: Count of positive customer reviews received that day
        expr: POSITIVE_REVIEWS
        data_type: NUMBER
      - name: NEGATIVE_REVIEWS
        description: Count of negative customer reviews received that day
        expr: NEGATIVE_REVIEWS
        data_type: NUMBER
    metrics:
      - name: TOTAL_REVENUE_UNITS
        description: Sum of all units sold
        expr: SUM(TOTAL_UNITS_SOLD)
        data_type: NUMBER
      - name: ONLINE_MIX_PERCENTAGE
        description: Percentage of sales from online channel
        expr: ROUND(SUM(ONLINE_UNITS) * 100.0 / NULLIF(SUM(TOTAL_UNITS_SOLD), 0), 1)
        data_type: FLOAT
      - name: AVG_SENTIMENT
        description: Average sentiment score across the period
        expr: AVG(AVG_DAILY_SENTIMENT)
        data_type: FLOAT
      - name: REVIEW_POSITIVITY_RATE
        description: Percentage of reviews that are positive
        expr: ROUND(SUM(POSITIVE_REVIEWS) * 100.0 / NULLIF(SUM(POSITIVE_REVIEWS) + SUM(NEGATIVE_REVIEWS), 0), 1)
        data_type: FLOAT

  - name: WEEKLY_PRODUCT_SCORECARD
    base_table:
      database: RETAIL_HOL_ICEBERG
      schema: GOLD_MANAGED_ICEBERG
      table: WEEKLY_PRODUCT_SCORECARD
    description: >-
      Weekly executive scorecard per product with aggregated KPIs.
      Each row is one product for one week starting Monday.
    dimensions:
      - name: PRODUCT_NAME
        description: Product name joins to DAILY_PRODUCT_PERFORMANCE.PRODUCT_NAME
        expr: PRODUCT_NAME
        data_type: VARCHAR
      - name: WORST_STOCK_STATUS
        description: Worst inventory health status observed during the week
        expr: WORST_STOCK_STATUS
        data_type: VARCHAR
    time_dimensions:
      - name: WEEK_START
        description: The Monday start date of the business week
        expr: WEEK_START
        data_type: DATE
    facts:
      - name: WEEKLY_UNITS_SOLD
        description: Total units sold during the week across all channels
        expr: WEEKLY_UNITS_SOLD
        data_type: NUMBER
      - name: WEEKLY_ONLINE_UNITS
        description: Units sold online during the week
        expr: WEEKLY_ONLINE_UNITS
        data_type: NUMBER
      - name: WEEKLY_STORE_UNITS
        description: Units sold in physical stores during the week
        expr: WEEKLY_STORE_UNITS
        data_type: NUMBER
      - name: ONLINE_MIX_PCT
        description: Percentage of weekly sales from online channel
        expr: ONLINE_MIX_PCT
        data_type: FLOAT
      - name: AVG_INVENTORY
        description: Average inventory position during the week
        expr: AVG_INVENTORY
        data_type: FLOAT
      - name: AVG_QUALITY_SCORE
        description: Average product quality score for the week
        expr: AVG_QUALITY_SCORE
        data_type: FLOAT
      - name: WEEKLY_SENTIMENT
        description: Average customer sentiment for the week
        expr: WEEKLY_SENTIMENT
        data_type: FLOAT
      - name: WEEKLY_POSITIVE
        description: Total positive reviews received during the week
        expr: WEEKLY_POSITIVE
        data_type: NUMBER
      - name: WEEKLY_NEGATIVE
        description: Total negative reviews received during the week
        expr: WEEKLY_NEGATIVE
        data_type: NUMBER
      - name: POSITIVE_REVIEW_PCT
        description: Percentage of weekly reviews that are positive
        expr: POSITIVE_REVIEW_PCT
        data_type: FLOAT
    metrics:
      - name: TOTAL_WEEKLY_VOLUME
        description: Sum of weekly units sold
        expr: SUM(WEEKLY_UNITS_SOLD)
        data_type: NUMBER
      - name: AVG_WEEKLY_SENTIMENT
        description: Average weekly sentiment
        expr: AVG(WEEKLY_SENTIMENT)
        data_type: FLOAT

  - name: CUSTOMER_VOICE_ANALYTICS
    base_table:
      database: RETAIL_HOL_ICEBERG
      schema: GOLD_MANAGED_ICEBERG
      table: CUSTOMER_VOICE_ANALYTICS
    description: >-
      Customer comments enriched with Cortex AI sentiment scoring
      auto-summarization and intent classification.
    dimensions:
      - name: PRODUCT_NAME
        description: Product the comment is about joins to DAILY_PRODUCT_PERFORMANCE.PRODUCT_NAME
        expr: PRODUCT_NAME
        data_type: VARCHAR
      - name: COMMENT_TITLE
        description: Short title of the customer comment
        expr: COMMENT_TITLE
        data_type: VARCHAR
      - name: COMMENT_BODY
        description: Full text of the customer comment
        expr: COMMENT_BODY
        data_type: VARCHAR
      - name: USER_NAME
        description: Username of the commenter
        expr: USER_NAME
        data_type: VARCHAR
      - name: CHANNEL
        description: Channel where comment was received web social or email
        expr: CHANNEL
        data_type: VARCHAR
      - name: AI_SUMMARY
        description: AI generated one line summary of the comment
        expr: AI_SUMMARY
        data_type: VARCHAR
      - name: CUSTOMER_INTENT
        description: AI classified intent praise complaint question suggestion or shipping_issue
        expr: CUSTOMER_INTENT
        data_type: VARCHAR
    time_dimensions:
      - name: COMMENT_DATE
        description: Date the comment was posted
        expr: COMMENT_DATE
        data_type: DATE
    facts:
      - name: SENTIMENT_SCORE
        description: AI sentiment score from negative 1 to positive 1
        expr: SENTIMENT_SCORE
        data_type: FLOAT
    metrics:
      - name: TOTAL_COMMENTS
        description: Total number of customer comments
        expr: COUNT(*)
        data_type: NUMBER
      - name: AVG_COMMENT_SENTIMENT
        description: Average sentiment score across comments
        expr: AVG(SENTIMENT_SCORE)
        data_type: FLOAT
      - name: COMPLAINT_RATE
        description: Percentage of comments classified as complaints
        expr: "ROUND(SUM(CASE WHEN CUSTOMER_INTENT = 'complaint' THEN 1 ELSE 0 END) * 100.0 / NULLIF(COUNT(*), 0), 1)"
        data_type: FLOAT

  - name: SILVER_INVENTORY
    base_table:
      database: RETAIL_HOL_ICEBERG
      schema: SILVER_MANAGED_ICEBERG
      table: SILVER_INVENTORY
    description: >-
      Inventory feed with warehouse geolocation. Each row is one SKU snapshot
      at one warehouse. Warehouses are in Los Angeles, Houston, New York,
      Chicago, and San Francisco. Use WAREHOUSE_LONGITUDE and WAREHOUSE_LATITUDE
      for spatial inventory queries. SKU_NAME joins to PRODUCT_NAME in other tables.
    dimensions:
      - name: SKU_NAME
        description: Product SKU name joins to DAILY_PRODUCT_PERFORMANCE.PRODUCT_NAME
        expr: SKU_NAME
        data_type: VARCHAR
      - name: SUPPLIER
        description: Supplier name PureEssence FragranceCo or AromaTech
        expr: SUPPLIER
        data_type: VARCHAR
      - name: STOCK_HEALTH
        description: Stock health CRITICAL LOW NORMAL or OVERSTOCKED
        expr: STOCK_HEALTH
        data_type: VARCHAR
    time_dimensions:
      - name: SNAPSHOT_DATE
        description: Date of the inventory snapshot
        expr: SNAPSHOT_DATE
        data_type: DATE
    facts:
      - name: QTY_ONLINE
        description: Quantity available for online fulfillment
        expr: QTY_ONLINE
        data_type: NUMBER
      - name: QTY_NORTHEAST
        description: Quantity at Northeast regional warehouse
        expr: QTY_NORTHEAST
        data_type: NUMBER
      - name: QTY_SOUTHEAST
        description: Quantity at Southeast regional warehouse
        expr: QTY_SOUTHEAST
        data_type: NUMBER
      - name: QTY_SOUTHWEST
        description: Quantity at Southwest regional warehouse
        expr: QTY_SOUTHWEST
        data_type: NUMBER
      - name: QTY_NORTHWEST
        description: Quantity at Northwest regional warehouse
        expr: QTY_NORTHWEST
        data_type: NUMBER
      - name: TOTAL_STOCK
        description: Total stock across all regions
        expr: TOTAL_STOCK
        data_type: NUMBER
      - name: QUALITY_SCORE
        description: Supplier quality score 0 to 100
        expr: QUALITY_SCORE
        data_type: FLOAT
      - name: WAREHOUSE_LONGITUDE
        description: Warehouse GPS longitude. Los Angeles is -118.24, Houston -95.37, New York -73.99, Chicago -87.63, San Francisco -122.42
        expr: WAREHOUSE_LONGITUDE
        data_type: FLOAT
      - name: WAREHOUSE_LATITUDE
        description: Warehouse GPS latitude. Los Angeles is 34.05, Houston 29.76, New York 40.75, Chicago 41.88, San Francisco 37.77
        expr: WAREHOUSE_LATITUDE
        data_type: FLOAT
    metrics:
      - name: TOTAL_INVENTORY_UNITS
        description: Sum of total stock across all records
        expr: SUM(TOTAL_STOCK)
        data_type: NUMBER
      - name: AVG_QUALITY
        description: Average quality score
        expr: AVG(QUALITY_SCORE)
        data_type: FLOAT

verified_queries:
  - name: top_selling_products
    question: What are the top 5 best-selling products?
    sql: |
      SELECT PRODUCT_NAME, SUM(TOTAL_UNITS_SOLD) AS TOTAL_UNITS
      FROM RETAIL_HOL_ICEBERG.GOLD_MANAGED_ICEBERG.DAILY_PRODUCT_PERFORMANCE
      GROUP BY PRODUCT_NAME ORDER BY TOTAL_UNITS DESC LIMIT 5
    verified_at: 1722211200
    verified_by: RETAIL_ANALYTICS_TEAM
  - name: critical_stock_products
    question: Which products have critical stock levels?
    sql: |
      SELECT DISTINCT PRODUCT_NAME, INVENTORY_POSITION, STOCK_HEALTH_STATUS
      FROM RETAIL_HOL_ICEBERG.GOLD_MANAGED_ICEBERG.DAILY_PRODUCT_PERFORMANCE
      WHERE STOCK_HEALTH_STATUS = 'CRITICAL'
      ORDER BY INVENTORY_POSITION ASC
    verified_at: 1722211200
    verified_by: SUPPLY_CHAIN_TEAM
  - name: customer_complaints
    question: What are customers complaining about?
    sql: |
      SELECT PRODUCT_NAME, COUNT(*) AS COMPLAINTS, AVG(SENTIMENT_SCORE) AS AVG_SENTIMENT
      FROM RETAIL_HOL_ICEBERG.GOLD_MANAGED_ICEBERG.CUSTOMER_VOICE_ANALYTICS
      WHERE CUSTOMER_INTENT = 'complaint'
      GROUP BY PRODUCT_NAME ORDER BY COMPLAINTS DESC
    verified_at: 1722211200
    verified_by: CX_TEAM
  - name: online_vs_store_mix
    question: What is the online vs store channel mix?
    sql: |
      SELECT PRODUCT_NAME,
             SUM(WEEKLY_ONLINE_UNITS) AS ONLINE_TOTAL,
             SUM(WEEKLY_STORE_UNITS) AS STORE_TOTAL,
             ROUND(SUM(WEEKLY_ONLINE_UNITS)*100.0/NULLIF(SUM(WEEKLY_UNITS_SOLD),0),1) AS ONLINE_PCT
      FROM RETAIL_HOL_ICEBERG.GOLD_MANAGED_ICEBERG.WEEKLY_PRODUCT_SCORECARD
      GROUP BY PRODUCT_NAME ORDER BY ONLINE_PCT DESC
    verified_at: 1722211200
    verified_by: ECOMMERCE_TEAM
  - name: inventory_by_warehouse_location
    question: What is the inventory distribution across warehouse locations?
    sql: |
      SELECT
        CASE
          WHEN ROUND(WAREHOUSE_LONGITUDE,2) = -118.24 THEN 'Los Angeles'
          WHEN ROUND(WAREHOUSE_LONGITUDE,2) = -95.37 THEN 'Houston'
          WHEN ROUND(WAREHOUSE_LONGITUDE,2) = -73.99 THEN 'New York'
          WHEN ROUND(WAREHOUSE_LONGITUDE,2) = -87.63 THEN 'Chicago'
          WHEN ROUND(WAREHOUSE_LONGITUDE,2) = -122.42 THEN 'San Francisco'
        END AS WAREHOUSE_CITY,
        COUNT(DISTINCT SKU_NAME) AS PRODUCTS_STOCKED,
        SUM(TOTAL_STOCK) AS TOTAL_INVENTORY
      FROM RETAIL_HOL_ICEBERG.SILVER_MANAGED_ICEBERG.SILVER_INVENTORY
      GROUP BY 1
      ORDER BY TOTAL_INVENTORY DESC
    verified_at: 1722211200
    verified_by: SUPPLY_CHAIN_TEAM
  - name: stock_health_by_warehouse
    question: Which warehouses have products at critical or low stock levels?
    sql: |
      SELECT
        CASE
          WHEN ROUND(WAREHOUSE_LONGITUDE,2) = -118.24 THEN 'Los Angeles'
          WHEN ROUND(WAREHOUSE_LONGITUDE,2) = -95.37 THEN 'Houston'
          WHEN ROUND(WAREHOUSE_LONGITUDE,2) = -73.99 THEN 'New York'
          WHEN ROUND(WAREHOUSE_LONGITUDE,2) = -87.63 THEN 'Chicago'
          WHEN ROUND(WAREHOUSE_LONGITUDE,2) = -122.42 THEN 'San Francisco'
        END AS WAREHOUSE_CITY,
        SKU_NAME,
        TOTAL_STOCK,
        STOCK_HEALTH
      FROM RETAIL_HOL_ICEBERG.SILVER_MANAGED_ICEBERG.SILVER_INVENTORY
      WHERE STOCK_HEALTH IN ('CRITICAL', 'LOW')
      ORDER BY TOTAL_STOCK ASC
    verified_at: 1722211200
    verified_by: SUPPLY_CHAIN_TEAM
$$
);

In [ ]:
%%sql -r dataframe_5
-- Verify the Semantic View was created
SHOW SEMANTIC VIEWS IN SCHEMA GOLD_MANAGED_ICEBERG;

**Semantic View created.** The agent now has a "business dictionary" — it knows what `STOCK_HEALTH_STATUS = 'CRITICAL'` means, how to compute `ONLINE_MIX_PERCENTAGE`, and which table to query for weekly vs. daily questions.

---
## Step 2: Create the Cortex Agent

The Semantic View defines *what the data means*. The **Cortex Agent** defines *how to interact with it*:
- A persona ("you are a retail analytics expert")
- Instructions ("always provide specific numbers", "flag products needing attention")
- A tool (`cortex_analyst_text_to_sql`) that generates SQL from natural language using the Semantic View

Think of it as: **Semantic View = brain** (knowledge), **Agent = personality** (behavior).

In [ ]:
%%sql -r dataframe_6
-- Create the Retail Intelligence Agent
CREATE OR REPLACE AGENT RETAIL_HOL_ICEBERG.GOLD_MANAGED_ICEBERG.RETAIL_ANALYST_AGENT
  COMMENT = 'Retail product intelligence agent - ask questions about sales, inventory, sentiment, and warehouse locations'
  PROFILE = '{"display_name": "Retail Analyst Agent"}'
  FROM SPECIFICATION
  $$
models:
  orchestration: auto

orchestration:
  budget:
    seconds: 30
    tokens: 16000

instructions:
  response: >-
    You are a retail analytics expert for a candle and fragrance company.
    You help merchandising, supply chain, marketing, and executive teams
    answer questions about product performance, inventory health, customer sentiment,
    and warehouse geolocation. Always provide specific numbers and percentages,
    compare to benchmarks when relevant (e.g., 80%+ positive review rate is good),
    flag products that need attention (critical stock, declining sentiment),
    and suggest actionable next steps when appropriate.
  orchestration: >-
    For time-based questions, default to the most recent data available.
    For location questions, map coordinates to city names:
    Los Angeles (-118.24, 34.05), Houston (-95.37, 29.76),
    New York (-73.99, 40.75), Chicago (-87.63, 41.88),
    San Francisco (-122.42, 37.77).
    Product portfolio: Frosted Juniper, Citrus Petal, Golden Amberleaf, Harvest Thyme,
    Lavender Honeycomb, Lemon Verbena Grove, Meadow Bloom, Peachwood Breeze,
    Rainleaf, Sea Salt Blossom, Spiced Oak & Apple, Vanilla Cedarlight.
    Channels: Online, Northeast stores, Northwest stores, Southeast stores, Southwest stores.
    Warehouse Locations: Los Angeles, Houston, New York, Chicago, San Francisco.
  sample_questions:
    - question: "What are the top 3 and bottom 3 products by total units sold?"
    - question: "Which products had critical or low stock health status?"
    - question: "Show me inventory distribution by warehouse city."

tools:
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: retail_analyst
      description: >-
        Answers questions about retail product performance, inventory health,
        customer sentiment, and warehouse geolocation by querying the
        RETAIL_PRODUCT_INTELLIGENCE semantic view.

tool_resources:
  retail_analyst:
    semantic_view: RETAIL_HOL_ICEBERG.GOLD_MANAGED_ICEBERG.RETAIL_PRODUCT_INTELLIGENCE
    execution_environment:
      type: warehouse
      warehouse: RETAIL_HOL_WH
$$;

In [ ]:
%%sql -r dataframe_7
-- Verify the agent is created
SHOW AGENTS IN SCHEMA GOLD_MANAGED_ICEBERG;

### Grant Access & Enable Snowflake CoWork

The agent exists, but only the owning role can use it. We need two more things:

1. **`GRANT USAGE ON AGENT`** — Controls who can invoke the agent (via the Agents UI, CoWork, or `DATA_AGENT_RUN`). Granting explicitly makes it reusable when you later add a `MERCHANDISING_ROLE` or `ANALYST_ROLE`.

2. **`CREATE SNOWFLAKE INTELLIGENCE` + `ADD AGENT`** — Snowflake CoWork is a shareable chat workspace. An agent isn't visible there automatically — it must be explicitly registered so it appears in the CoWork agent picker for your whole team.

In [ ]:
%%sql -r dataframe_8
USE ROLE ACCOUNTADMIN;
-- 1. Ensure the CoWork (Snowflake Intelligence) object exists
CREATE SNOWFLAKE INTELLIGENCE IF NOT EXISTS SNOWFLAKE_INTELLIGENCE_OBJECT_DEFAULT;

-- 2. Grant USAGE so the role can interact with the agent in CoWork
GRANT USAGE ON AGENT RETAIL_HOL_ICEBERG.GOLD_MANAGED_ICEBERG.RETAIL_ANALYST_AGENT
  TO ROLE RETAIL_HOL_ROLE;
  
-- 3. Add the agent to the CoWork object so it appears in the dropdown
ALTER SNOWFLAKE INTELLIGENCE SNOWFLAKE_INTELLIGENCE_OBJECT_DEFAULT 
  ADD AGENT RETAIL_HOL_ICEBERG.GOLD_MANAGED_ICEBERG.RETAIL_ANALYST_AGENT;

---
## Step 3: Talk to Your Data — Retail Questions

The agent is live. Let's ask it real retail questions — the same questions your merchandising, supply chain, marketing, and executive teams ask every day.

### Two Ways to Talk to the Agent

### Option A (Recommended) — Snowflake CoWork Chat UI (interactive, no code)

> **Tip — Duplicate your browser tab first!**  
> Right-click this browser tab and select **Duplicate**
> This opens Snowsight in a second tab so you can keep this notebook open side-by-side  
> without navigating away. No more back-and-forth!

**In the new (duplicated) tab, follow these steps:**

1. In the left navigation panel, expand **AI & ML**
2. Click **Snowflake CoWork**
3. In the CoWork interface, click **Capabilities** and then **Agents** on the left menu
4. Select the agent **`RETAIL_ANALYST_AGENT`**
5. Start chatting — type any retail question in the message box and hit Enter

Try questions like:
- *"What are my top selling products by region?"*
- *"Which SKUs have critical stock levels?"*
- *"Show me average sentiment by product category"*

The agent uses the Semantic View you just created to translate natural language into SQL against your Gold Iceberg tables.

---

#### Option B (For Application Integration) — `SNOWFLAKE.CORTEX.DATA_AGENT_RUN` (programmatic, in SQL):

```
**Note:** If you encounter the message  
"I've reached the time limit for this request. The analysis may be incomplete. Would you like me to continue working on this?"
This is typically due to network latency or connection timeout — not an issue with your query.  

We recommend switching to **Option A (Snowflake CoWork in Snowsight)** for the most reliable conversational AI experience.

```

Run the agent from a SQL cell — useful for automation, notebooks, or embedding in pipelines. This is what we use below.

> [DATA_AGENT_RUN Reference](https://docs.snowflake.com/en/sql-reference/functions/data_agent_run-snowflake-cortex)


### Merchandising Team: "What's selling and what's not?"

In [ ]:
%%sql -r dataframe_9
-- Q1: Merchandising — Top and bottom performers
WITH raw AS (
  SELECT TRY_PARSE_JSON(
    SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
      'RETAIL_HOL_ICEBERG.GOLD_MANAGED_ICEBERG.RETAIL_ANALYST_AGENT',
      $${
        "messages": [
          {
            "role": "user",
            "content": [
              { "type": "text", "text": "What are the top 3 and bottom 3 products by total units sold? Include their positive review percentage." }
            ]
          }
        ]
      }$$,
      TRUE
    )
  ) AS resp
)
SELECT f.value:text::VARCHAR AS ANSWER
FROM raw, LATERAL FLATTEN(input => resp:content) f
WHERE f.value:type::VARCHAR = 'text';


```
**Note:** If you encounter the message  
"I've reached the time limit for this request. The analysis may be incomplete. Would you like me to continue working on this?"
This is typically due to network latency or connection timeout — not an issue with your query.  

We recommend switching to **🔵 Option A (Snowflake CoWork  in Snowsight)** for the most reliable conversational AI experience.

```

### Supply Chain Team: "Do I need to reorder anything?"

In [ ]:
%%sql -r dataframe_10
-- Q2: Supply Chain — Stock health and reorder alert
WITH raw AS (
  SELECT TRY_PARSE_JSON(
    SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
      'RETAIL_HOL_ICEBERG.GOLD_MANAGED_ICEBERG.RETAIL_ANALYST_AGENT',
      $${
        "messages": [
          {
            "role": "user",
            "content": [
              { "type": "text", "text": "Which products had critical or low stock health status? How many days were they at that level?" }
            ]
          }
        ]
      }$$,
      TRUE
    )
  ) AS resp
)
SELECT f.value:text::VARCHAR AS ANSWER
FROM raw, LATERAL FLATTEN(input => resp:content) f
WHERE f.value:type::VARCHAR = 'text';

### Marketing Team: "How do customers feel about our products?"

In [ ]:
%%sql -r dataframe_12
-- Q3: Marketing — Sentiment analysis and customer voice
WITH raw AS (
  SELECT TRY_PARSE_JSON(
    SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
      'RETAIL_HOL_ICEBERG.GOLD_MANAGED_ICEBERG.RETAIL_ANALYST_AGENT',
      $${
        "messages": [
          {
            "role": "user",
            "content": [
              { "type": "text", "text": "Show me the products with the worst customer sentiment. What are customers complaining about? Break down by intent type." }
            ]
          }
        ]
      }$$,
      TRUE
    )
  ) AS resp
)
SELECT f.value:text::VARCHAR AS ANSWER
FROM raw, LATERAL FLATTEN(input => resp:content) f
WHERE f.value:type::VARCHAR = 'text';

### Executive Leadership: "Give me the weekly business review"

In [ ]:
%%sql -r dataframe_14
-- Q4: Executive — Weekly business review
WITH raw AS (
  SELECT TRY_PARSE_JSON(
    SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
      'RETAIL_HOL_ICEBERG.GOLD_MANAGED_ICEBERG.RETAIL_ANALYST_AGENT',
      $${
        "messages": [
          {
            "role": "user",
            "content": [
              { "type": "text", "text": "Give me a weekly business review: total units sold per week, trending sentiment, and flag any products with declining performance over the last 4 weeks." }
            ]
          }
        ]
      }$$,
      TRUE
    )
  ) AS resp
)
SELECT f.value:text::VARCHAR AS ANSWER
FROM raw, LATERAL FLATTEN(input => resp:content) f
WHERE f.value:type::VARCHAR = 'text';

### Quality & Operations: "Are there supplier quality issues?"

In [ ]:
%%sql -r dataframe_15
-- Q5: Operations — Quality score correlation with sentiment
WITH raw AS (
  SELECT TRY_PARSE_JSON(
    SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
      'RETAIL_HOL_ICEBERG.GOLD_MANAGED_ICEBERG.RETAIL_ANALYST_AGENT',
      $${
        "messages": [
          {
            "role": "user",
            "content": [
              { "type": "text", "text": "Is there a correlation between product quality score and customer sentiment? Show me products with quality below 90 and their average sentiment." }
            ]
          }
        ]
      }$$,
      TRUE
    )
  ) AS resp
)
SELECT f.value:text::VARCHAR AS ANSWER
FROM raw, LATERAL FLATTEN(input => resp:content) f
WHERE f.value:type::VARCHAR = 'text';

---
## More Questions to Explore

### Here are 20+ additional questions across retail functions. 
### 🔵 Ask them in the Snowflake CoWork  in Snowsight 🔵

### Merchandising & Assortment
| # | Question |
|---|----------|
| 1 | Which products have the highest sell-through rate? |
| 2 | What products should I consider discontinuing based on low sales and poor sentiment? |
| 3 | Are there seasonal patterns — which products peak in Q4 vs Q1? |
| 4 | Compare Frosted Juniper to Vanilla Cedarlight across all KPIs |

### Supply Chain & Inventory
| # | Question |
|---|----------|
| 5 | What is the days-of-supply for each product based on current inventory and daily sales? |
| 6 | Which products are consistently overstocked? Can we reduce order quantities? |
| 7 | Show me inventory position vs sales velocity — who needs rebalancing? |
| 8 | Which supplier (based on quality score) delivers the most consistent quality? |

### Customer Experience & Sentiment
| # | Question |
|---|----------|
| 9 | What percentage of customer comments are complaints vs praise? |
| 10 | Which channel (web, social, email) has the most negative sentiment? |
| 11 | Show me the actual comment text for the most negative feedback this month |
| 12 | Are shipping complaints correlated with any specific product? |

### E-Commerce & Channel Strategy
| # | Question |
|---|----------|
| 13 | Which products have the fastest-growing online penetration? |
| 14 | Is there a product better suited for stores-only vs online-only? |
| 15 | What's the weekly trend in online mix percentage? |
| 16 | Compare online vs store performance for products with quality score > 95 |

### Executive & Strategic
| # | Question |
|---|----------|
| 17 | Give me a one-page summary: total units, top product, biggest risk, biggest opportunity |
| 18 | Which products improved sentiment month-over-month? |
| 19 | What's our overall review positivity rate and how does it trend? |
| 20 | Rank all 12 products by a composite score of sales + sentiment + quality |

---
## Recap: The Complete Journey

Across Notebooks , you built an end-to-end system — from raw PostgreSQL rows to conversational business intelligence:

```
┌─────────────────────────────────────────────────────────────────────────────────┐
│                         INGESTION TO INSIGHT IN MINUTES                          │
├─────────────────────────────────────────────────────────────────────────────────┤
│                                                                                 │
│  ┌──────────┐    ┌──────────────┐    ┌────────────────┐    ┌────────────────┐  │
│  │PostgreSQL│───▶│Iceberg Bronze│───▶│Dynamic Iceberg │───▶│ Semantic View  │  │
│  │  (OLTP)  │CDC │   Tables     │ DT │  Silver/Gold   │    │ + Cortex Agent │  │
│  └──────────┘    └──────────────┘    └────────────────┘    └───────┬────────┘  │
│    OpenFlow         Zero-code          ADAPTIVE refresh       Natural Language  │
│    Real-time        Open format        Frozen Regions          Ask anything     │
│                                        Cortex AI enrich                         │
│                                                                       │        │
│                                                                       ▼        │
│                                                              ┌────────────────┐│
│                                                              │  Business User ││
│                                                              │  Asks Question ││
│                                                              │  Gets Answer   ││
│                                                              └────────────────┘│
│                                                                                 │
├─────────────────────────────────────────────────────────────────────────────────┤
│  SNOWFLAKE DIFFERENTIATION:                                                     │
│  ─────────────────────────                                                      │
│  1. All-in-one platform — no Kafka, no Airflow, no dbt, no external ML          │
│  2. Open format (Iceberg) — no vendor lock-in at any layer                      │
│  3. Declarative everything — DTs, Semantic Views, Agents = SQL DDL              │
│  4. Built-in AI — Cortex functions run inside the engine, no API calls          │
│  5. Zero-ops — ADAPTIVE refresh, Frozen Regions = self-managing                 │
│  6. Unified governance — one platform, one security model, one lineage          │
└─────────────────────────────────────────────────────────────────────────────────┘
```

### How Quickly Can You Create Value?

| Step | What | How Long | Traditional Approach |
|------|------|----------|---------------------|
| 1 | Connect source (PostgreSQL) | 5 min | Days (Kafka + Debezium + infra) |
| 2 | Transform raw → Silver/Gold | 5 min | Hours (dbt + scheduler + testing) |
| 3 | Add AI enrichment | 0 min (in DT) | Weeks (ML pipeline + model serving) |
| 4 | Build semantic layer | 5 min | Days (LookML / dbt metrics layer) |
| 5 | Enable NL querying | 2 min | Months (custom NLP + fine-tuning) |
| **Total** | **End-to-end** | **~15 min** | **Weeks to months** |

### What's Next from Here?

1. **Snowflake Intelligence (CoWork)** — Your agent is already registered. Go to AI & ML → Snowflake CoWork and share it with the team.
2. **Schedule Alerts** — Use Snowflake Tasks to run agent queries nightly and alert on anomalies.

---

**Congratulations!** You've gone from a PostgreSQL database to a conversational AI that any business user can query in plain English. No Kafka, no Airflow, no dbt, no external ML, no custom NLP. Just using Snowflake AI Data Platform.

**Optional**: Proceed to notebook `05-retail-hol-incremental-data-generation.ipynb` to generate incremental data changes (inserts, updates, and deletes) in your Postgres source. This simulates real-world CDC activity and lets you observe how changes flow end-to-end — from Postgres through Openflow into your Iceberg tables, and ultimately through the Snowflake Cortex Agent in Notebook 04.